In [1]:
import os
import urllib

import nglview
import numpy as np
from openff.toolkit import ForceField, Molecule, Topology
from openff.units import unit

from openff.interchange import Interchange
from openff.interchange.components._packmol import UNIT_CUBE, pack_box
from openff.interchange.drivers import get_openmm_energies
from openff.interchange.drivers.all import get_summary_data
from rdkit import Chem

In [4]:
# # Binder
# pdb_file = os.path.abspath("binders/seq6_binder/protein_seq6_binder_fixed_H.pdb")
# sdf_file = "binders/seq6_binder/ligand_seq6_binder.sdf"

# Nonbinder
pdb_file = os.path.abspath("nonbinders/seq1_nb/protein_seq1_nb_fixed_H.pdb")
sdf_file = "nonbinders/seq1_nb/ligand_seq1_nb.sdf"

## Prep components - protein+ligand

### Ligand

In [5]:
#ligand = Molecule.from_file("binders/seq6_binder/ligand_seq6_binder.sdf") # Binder
ligand = Molecule.from_file("nonbinders/seq1_nb/ligand_seq1_nb.sdf") # Nonbinder

In [6]:
ligand.name = "LIG"
ligand.visualize(backend="nglview")

NGLWidget()

In [7]:
ligand_intrcg = Interchange.from_smirnoff(
    force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=[ligand],
)

### Protein

In [8]:
protein_full = Topology.from_pdb(pdb_file)
protein_full.n_molecules

1

In [9]:
protein = protein_full.molecule(0)
protein.name = "protein"
protein.visualize(backend="nglview")

NGLWidget()

In [10]:
ff14sb = ForceField("ff14sb_off_impropers_0.0.3.offxml")

In [11]:
protein_intrcg = Interchange.from_smirnoff(
    force_field=ff14sb,
    topology=protein.to_topology(),
)

### Dock ligand + protein

In [12]:
docked_intrcg = protein_intrcg.combine(ligand_intrcg) # dock

# visualize
w = docked_intrcg.visualize()
w.clear_representations()
w.add_representation(
    "licorice",
    radius=0.1,
    selection=[*range(protein_intrcg.topology.n_atoms)],
)
w.add_representation(
    "spacefill",
    selection=[*range(protein_intrcg.topology.n_atoms, docked_intrcg.topology.n_atoms)],
)
w

/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/operations/_combine.py:104: InterchangeCombinationWarning: Interchange object combination is complex and may produce strange results outside of use cases it has been tested in. Use with caution and thoroughly validate results!
  warnings.warn(
/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/operations/_combine.py:84: InterchangeCombinationWarning: Found electrostatics 1-4 scaling factors of 5/6 with slightly different rounding (0.833333 and 0.8333333333). This likely stems from OpenFF using more digits in rounding 1/1.2. The value of 0.8333333333 will be used, which may or may not introduce small errors. 
  warnings.warn(


NGLWidget()

Check partial charge of protein+ligand complex

In [13]:
total_charge = round(sum(docked_intrcg["Electrostatics"].charges.values()), 3)

assert total_charge == protein.total_charge + ligand.total_charge, (
    f"Total charge of the system is {total_charge}, not {protein.total_charge + ligand.total_charge}"
)
total_charge

<Quantity(-3.0, 'elementary_charge')>

### Solvent

In [14]:
total_charge_e = float(total_charge.m)

if total_charge_e < 0:
    counterion = Molecule.from_smiles("[Na+]")
    counterion.name = "NA"
    n_counterions = int(round(abs(total_charge_e)))
elif total_charge_e > 0:
    counterion = Molecule.from_smiles("[Cl-]")
    counterion.name = "CL"
    n_counterions = int(round(abs(total_charge_e)))
else:
    counterion = None
    n_counterions = 0
    
water = Molecule.from_smiles("O")
water.name = "SOL"
water.generate_conformers(n_conformers=1)

# Compute a reasonable box
xyz = protein.conformers[0].to(unit.nanometer).m
centroid = xyz.mean(axis=0)
protein_radius_nm = np.sqrt(((xyz - centroid) ** 2).sum(axis=1).max())

buffer_nm = 2.0
box_length_nm = 2.0 * protein_radius_nm + buffer_nm
box_vectors = np.eye(3) * box_length_nm * unit.nanometer

# ----------------------------
# Compute water count for ~1 g/mL
# ----------------------------
V_box_nm3 = box_length_nm ** 3

# crude displaced volume estimate (protein roughly spherical)
V_solute_nm3 = (4.0 / 3.0) * np.pi * (protein_radius_nm ** 3)

waters_per_nm3 = 33.4  # ~1 g/mL at 300 K
n_water = int(waters_per_nm3 * max(V_box_nm3 - V_solute_nm3, 0.0))

# (Optional) add a little cushion if you find voids / low density
# n_water = int(1.05 * n_water)

print(f"Box length: {box_length_nm:.3f} nm")
print(f"Packing waters: {n_water}, counterions: {n_counterions}")

# ----------------------------
# Pack the box
# ----------------------------
molecules = [water]
number_of_copies = [n_water]

if counterion is not None and n_counterions > 0:
    molecules.append(counterion)
    number_of_copies.append(n_counterions)

packed_topology = pack_box(
    solute=docked_intrcg.topology,
    molecules=molecules,
    number_of_copies=number_of_copies,
    box_vectors=box_vectors,
    center_solute=True,
    tolerance=2.0 * unit.angstrom,
    working_directory="packmol_solv",
    retain_working_files=True,
)

# (Optional) quick sanity prints
print("Box length (nm):", box_length_nm)
print("Total molecules packed:", packed_topology.n_molecules)

Box length: 8.229 nm
Packing waters: 14386, counterions: 3
Box length (nm): 8.229292697928877
Total molecules packed: 14391


In [15]:
packed_topology.to_file("packed_seq1_nb.pdb") # save
nglview.show_structure_file("packed_seq1_nb.pdb") # visualize

NGLWidget()

In [16]:
topology_molecules = [water] * n_water
if counterion is not None:
    topology_molecules += [counterion] * n_counterions

In [17]:
# create Interchange for box of water
water_intrcg = Interchange.from_smirnoff(
    force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=topology_molecules,
)

### Put everything together 
Dock protein+ligand complex with the solvent

In [18]:
system_intrcg = docked_intrcg.combine(water_intrcg)
system_intrcg.positions = packed_topology.get_positions()
system_intrcg.box = packed_topology.box_vectors

/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/operations/_combine.py:183: InterchangeCombinationWarning: Setting positions to None because one or both objects added together were missing positions.
  warnings.warn(


In [19]:
w = system_intrcg.visualize()
w.clear_representations()
# Protein rep
w.add_representation(
    "licorice",
    radius=0.2,
    selection=[*range(protein_intrcg.topology.n_atoms)],
)
# Ligand rep
w.add_representation(
    "spacefill",
    selection=[*range(protein_intrcg.topology.n_atoms, docked_intrcg.topology.n_atoms)],
)
# Water rep
w.add_representation(
    "licorice",
    radius=0.1,
    selection=[*range(docked_intrcg.topology.n_atoms, system_intrcg.topology.n_atoms)],
)
w

NGLWidget()

## Export to GROMACS

In [20]:
system_intrcg.to_gromacs(prefix="seq1_nb", monolithic=False)

/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/components/mdconfig.py:503: UserWarning: Ambiguous failure while processing constraints. Constraining h-bonds as a stopgap.
  warnings.warn(
